# CX Assist : Notebook 02: Policy Retrieval with Citations

This notebook builds a model-independent retrieval layer for any CX case. It loads policies and cases from S3, splits policies by Markdown section, creates sparse TF-IDF vectors, ranks chunks using the case context, and returns source-grounded citations.


## 1. Imports and configuration

Add `scikit-learn` to `requirement.txt` before running this notebook.


In [1]:
from __future__ import annotations

import io
import os
import re
from dataclasses import asdict, dataclass

import boto3
import pandas as pd
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

AWS_PROFILE = os.getenv("AWS_PROFILE", "CXASSIST")
AWS_REGION = os.getenv("AWS_REGION", "ap-south-1")
S3_BUCKET = os.getenv("S3_BUCKET", "rahulcxassistdemo")
S3_PREFIX = os.getenv("S3_PREFIX", "cx-copilot/dev").strip("/")

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")

print(f"Connected configuration: s3://{S3_BUCKET}/{S3_PREFIX}/")


Connected configuration: s3://rahulcxassistdemo/cx-copilot/dev/


## 2. Load cases and policy documents from S3


In [2]:
def read_s3_bytes(relative_path: str) -> bytes:
    key = f"{S3_PREFIX}/{relative_path.lstrip('/')}"
    response = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return response["Body"].read()


def read_s3_csv(filename: str) -> pd.DataFrame:
    return pd.read_csv(io.BytesIO(read_s3_bytes(f"structured/{filename}")))


def list_policy_keys() -> list[str]:
    policy_prefix = f"{S3_PREFIX}/policies/"
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=policy_prefix):
        keys.extend(
            item["Key"] for item in page.get("Contents", [])
            if item["Key"].endswith(".md")
        )
    return sorted(keys)


cases = read_s3_csv("cases.csv")
policy_keys = list_policy_keys()
policy_documents = {
    key.rsplit("/", 1)[-1]: s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read().decode("utf-8")
    for key in policy_keys
}

print(f"Loaded {len(cases)} cases and {len(policy_documents)} policies.")
print(*policy_documents.keys(), sep="\n- ")


Loaded 5 cases and 4 policies.
billing_adjustment_policy.md
- duplicate_charge_policy.md
- escalation_and_approval_policy.md
- maintenance_downtime_policy.md


## 3. Parse policy metadata and split documents by section

Heading-aware chunks preserve policy meaning better than arbitrary character windows for these short documents.


In [3]:
@dataclass(frozen=True)
class PolicyChunk:
    chunk_id: str
    document_id: str
    document_name: str
    filename: str
    section: str
    version: str
    effective_date: str
    owner: str
    content: str
    source_key: str


def metadata_value(text: str, label: str) -> str:
    match = re.search(rf"\*\*{re.escape(label)}:\*\*\s*(.+)", text)
    return match.group(1).strip() if match else "Unknown"


def chunk_policy(filename: str, text: str) -> list[PolicyChunk]:
    title_match = re.search(r"^#\s+(.+)$", text, flags=re.MULTILINE)
    title = title_match.group(1).strip() if title_match else filename
    document_id = metadata_value(text, "Document ID")
    version = metadata_value(text, "Version")
    effective_date = metadata_value(text, "Effective date")
    owner = metadata_value(text, "Owner")
    sections = re.split(r"^##\s+", text, flags=re.MULTILINE)[1:]
    chunks = []

    for position, raw_section in enumerate(sections, start=1):
        lines = raw_section.strip().splitlines()
        section = lines[0].strip()
        content = "\n".join(lines[1:]).strip()
        if not content:
            continue
        chunks.append(PolicyChunk(
            chunk_id=f"{document_id}-S{position:02d}",
            document_id=document_id,
            document_name=title,
            filename=filename,
            section=section,
            version=version,
            effective_date=effective_date,
            owner=owner,
            content=content,
            source_key=f"{S3_PREFIX}/policies/{filename}",
        ))
    return chunks


policy_chunks = [
    chunk
    for filename, text in policy_documents.items()
    for chunk in chunk_policy(filename, text)
]

chunk_table = pd.DataFrame([asdict(chunk) for chunk in policy_chunks])
chunk_table[["chunk_id", "document_id", "document_name", "section"]]


,chunk_id,document_id,document_name,section
0,POL-BILL-001-S01,POL-BILL-001,Billing Adjustment Policy,Purpose
1,POL-BILL-001-S02,POL-BILL-001,Billing Adjustment Policy,Required evidence
2,POL-BILL-001-S03,POL-BILL-001,Billing Adjustment Policy,Adjustment calculation
3,POL-BILL-001-S04,POL-BILL-001,Billing Adjustment Policy,Approval
4,POL-BILL-001-S05,POL-BILL-001,Billing Adjustment Policy,Customer communication
5,POL-BILL-002-S01,POL-BILL-002,Duplicate Charge Policy,Identification
6,POL-BILL-002-S02,POL-BILL-002,Duplicate Charge Policy,Required checks
7,POL-BILL-002-S03,POL-BILL-002,Duplicate Charge Policy,Resolution
8,POL-BILL-002-S04,POL-BILL-002,Duplicate Charge Policy,Exceptions
9,POL-CX-004-S01,POL-CX-004,CX Escalation and Approval Policy,Escalation triggers


## 4. Build the vector index

This baseline uses TF-IDF vectors and cosine similarity. It is fast, local, explainable, and requires no embedding API. A dense embedding adapter can later replace the vectorizer without changing the citation contract.


In [4]:
search_texts = [
    " ".join([
        chunk.document_name, chunk.section, chunk.owner, chunk.content
    ])
    for chunk in policy_chunks
]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
)
policy_matrix = vectorizer.fit_transform(search_texts)

print(f"Indexed {policy_matrix.shape[0]} chunks with {policy_matrix.shape[1]} features.")


Indexed 17 chunks with 606 features.


## 5. Hybrid retrieval: vector score plus business routing

The issue-type hints act as a small routing boost, not a hard filter. Relevant policies outside the hint list can still be retrieved.


In [5]:
POLICY_HINTS = {
    "Billing Dispute": {"POL-BILL-001", "POL-SVC-003"},
    "Duplicate Charge": {"POL-BILL-001", "POL-BILL-002"},
    "Billing Explanation": {"POL-BILL-001"},
    "Payment Reconciliation": {"POL-BILL-001", "POL-CX-004"},
    "Roadside Assistance": {"POL-CX-004"},
}


def build_case_query(case: pd.Series) -> str:
    return " ".join(str(case.get(field, "")) for field in [
        "issue_type", "priority", "subject", "description"
    ])


def retrieve_policy_chunks(
    query: str, issue_type: str | None = None, top_k: int = 4
) -> list[dict]:
    query_vector = vectorizer.transform([query])
    similarity = cosine_similarity(query_vector, policy_matrix).flatten()
    hinted_ids = POLICY_HINTS.get(issue_type or "", set())
    results = []

    for index, chunk in enumerate(policy_chunks):
        routing_boost = 0.15 if chunk.document_id in hinted_ids else 0.0
        final_score = min(float(similarity[index]) + routing_boost, 1.0)
        item = asdict(chunk)
        item.update({
            "similarity_score": round(float(similarity[index]), 4),
            "routing_boost": routing_boost,
            "retrieval_score": round(final_score, 4),
            "citation": f"{chunk.document_id} v{chunk.version} — {chunk.section}",
        })
        results.append(item)

    return sorted(
        results, key=lambda item: item["retrieval_score"], reverse=True
    )[:top_k]


def retrieve_policies_for_case(case_id: str, top_k: int = 4) -> list[dict]:
    normalized_id = case_id.strip().upper()
    matches = cases.loc[cases["case_id"] == normalized_id]
    if matches.empty:
        raise ValueError(f"Case '{normalized_id}' was not found.")
    case = matches.iloc[0]
    return retrieve_policy_chunks(
        query=build_case_query(case),
        issue_type=case["issue_type"],
        top_k=top_k,
    )


## 6. Retrieve policies for any selected case


In [6]:
available_case_ids = sorted(cases["case_id"].tolist())
selected_case_id = available_case_ids[0]  # Change to any available case ID.

retrieved = retrieve_policies_for_case(selected_case_id, top_k=4)
display(pd.DataFrame(retrieved)[[
    "document_id", "document_name", "section",
    "similarity_score", "routing_boost", "retrieval_score", "citation"
]])

for result in retrieved:
    print(f"\n[{result['citation']}] score={result['retrieval_score']}")
    print(result["content"])


,document_id,document_name,section,similarity_score,routing_boost,retrieval_score,citation
0,POL-SVC-003,Maintenance Downtime Credit Policy,Eligibility,0.2456,0.15,0.3956,POL-SVC-003 v2.1 — Eligibility
1,POL-BILL-001,Billing Adjustment Policy,Adjustment calculation,0.1536,0.15,0.3036,POL-BILL-001 v1.2 — Adjustment calculation
2,POL-BILL-001,Billing Adjustment Policy,Required evidence,0.0704,0.15,0.2204,POL-BILL-001 v1.2 — Required evidence
3,POL-SVC-003,Maintenance Downtime Credit Policy,Approval and communication,0.0454,0.15,0.1954,POL-SVC-003 v2.1 — Approval and communication



[POL-SVC-003 v2.1 — Eligibility] score=0.3956
A recurring-charge adjustment may be considered when an active covered unit is unavailable for more than 24 consecutive hours due to a verified unscheduled repair. Scheduled preventive maintenance, inspections, customer-caused damage, and downtime caused by customer delay are not automatically eligible.

[POL-BILL-001 v1.2 — Adjustment calculation] score=0.3036
When an applicable policy authorizes a prorated adjustment, calculate the daily rate as the recurring monthly charge divided by the number of calendar days in the billing period. Multiply the daily rate by the number of eligible whole downtime days. Taxes and unrelated fees are excluded unless separately approved.

[POL-BILL-001 v1.2 — Required evidence] score=0.2204
The reviewer must verify the customer, contract, unit, invoice, billing period, payment status, and related service or case records. If required evidence is unavailable or inconsistent, the case must remain open and the

## 7. Validate retrieval across every case


In [7]:
EXPECTED_DOCUMENTS = {
    "CASE-3021": {"POL-BILL-001", "POL-SVC-003"},
    "CASE-3022": {"POL-BILL-002"},
    "CASE-3023": {"POL-BILL-001"},
    "CASE-3024": {"POL-BILL-001", "POL-CX-004"},
    "CASE-3025": {"POL-CX-004"},
}

evaluation_rows = []
for case_id, expected_ids in EXPECTED_DOCUMENTS.items():
    results = retrieve_policies_for_case(case_id, top_k=6)
    retrieved_ids = {item["document_id"] for item in results}
    matched = expected_ids & retrieved_ids
    evaluation_rows.append({
        "case_id": case_id,
        "expected_documents": sorted(expected_ids),
        "retrieved_documents": sorted(retrieved_ids),
        "matched_documents": sorted(matched),
        "all_expected_retrieved": expected_ids.issubset(retrieved_ids),
    })

evaluation = pd.DataFrame(evaluation_rows)
assert evaluation["all_expected_retrieved"].all(), evaluation
evaluation


,case_id,expected_documents,retrieved_documents,matched_documents,all_expected_retrieved
0,CASE-3021,"[POL-BILL-001, POL-SVC-003]","[POL-BILL-001, POL-SVC-003]","[POL-BILL-001, POL-SVC-003]",True
1,CASE-3022,[POL-BILL-002],"[POL-BILL-001, POL-BILL-002]",[POL-BILL-002],True
2,CASE-3023,[POL-BILL-001],"[POL-BILL-001, POL-BILL-002]",[POL-BILL-001],True
3,CASE-3024,"[POL-BILL-001, POL-CX-004]","[POL-BILL-001, POL-CX-004]","[POL-BILL-001, POL-CX-004]",True
4,CASE-3025,[POL-CX-004],"[POL-BILL-001, POL-CX-004, POL-SVC-003]",[POL-CX-004],True


## 8. Create an LLM-ready evidence package

Only retrieved policy sections—not the entire knowledge base—should be sent to the investigation model.


In [8]:
def format_policy_context(results: list[dict]) -> str:
    blocks = []
    for item in results:
        blocks.append(
            f"SOURCE: {item['citation']}\n"
            f"OWNER: {item['owner']}\n"
            f"EFFECTIVE DATE: {item['effective_date']}\n"
            f"CONTENT: {item['content']}"
        )
    return "\n\n---\n\n".join(blocks)


policy_context = format_policy_context(retrieved)
print(policy_context)


SOURCE: POL-SVC-003 v2.1 — Eligibility
OWNER: Fleet Maintenance and Customer Experience
EFFECTIVE DATE: 2026-03-01
CONTENT: A recurring-charge adjustment may be considered when an active covered unit is unavailable for more than 24 consecutive hours due to a verified unscheduled repair. Scheduled preventive maintenance, inspections, customer-caused damage, and downtime caused by customer delay are not automatically eligible.

---

SOURCE: POL-BILL-001 v1.2 — Adjustment calculation
OWNER: Customer Experience and Billing Operations
EFFECTIVE DATE: 2026-01-01
CONTENT: When an applicable policy authorizes a prorated adjustment, calculate the daily rate as the recurring monthly charge divided by the number of calendar days in the billing period. Multiply the daily rate by the number of eligible whole downtime days. Taxes and unrelated fees are excluded unless separately approved.

---

SOURCE: POL-BILL-001 v1.2 — Required evidence
OWNER: Customer Experience and Billing Operations
EFFECTIVE 

## Completion criteria

Notebook 02 is complete when all four documents load from S3, policy metadata is parsed, every case retrieves its expected documents, citations include policy ID/version/section, and an LLM-ready evidence package is produced. Notebook 03 will add the provider-switching model factory and connectivity tests.
